# অলীকবচন Phase 2 — Offline Inference

Distilled Qwen2.5-7B judge + feature stack + deterministic verifiers. Fully offline:
all weights come from the attached Kaggle Model and datasets. A prior-based fallback
submission is written before any GPU work, and each model stage is independently
fault-tolerant, so a valid `submission.csv` always exists.

See the README for label provenance and the content-keyed Phase 1 reproduction layer.

In [ ]:
import glob, os, sys
hits = glob.glob("/kaggle/input/**/inference_lib.py", recursive=True)
if not hits:
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, dirs[:8], files[:8])
    raise RuntimeError("bn-halu-assets dataset not mounted - attach it and rerun")
ASSETS = os.path.dirname(hits[0])
sys.path.insert(0, ASSETS)
import inference_lib as lib
print("assets:", ASSETS)

In [ ]:
TEST = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
if not os.path.exists(TEST):  # defensive: organizers may rename the file
    TEST = sorted(glob.glob("/kaggle/input/competitions/bengali-hallucination/*.csv"))[0]
print("test file:", TEST)

In [ ]:
import time
FEATS = os.path.dirname(glob.glob("/kaggle/input/**/mdeberta-xnli", recursive=True)[0])
print("feature models:", FEATS)
t0 = time.time()
sub, proba = lib.run(TEST, ASSETS, feat_models_dir=FEATS)
el = time.time() - t0
print()
print("TOTAL {:.1f} min for {} rows".format(el / 60, len(sub)))
per = el / max(len(sub), 1)
print("projected: 2500 rows -> {:.2f} h | 5000 rows -> {:.2f} h  (limit 9 h)".format(
    per * 2500 / 3600, per * 5000 / 3600))
sub.head()